In [1]:
import pandas as pd
# Se importa la clase Helper del módulo helper.helper
#from helper.helper import Helper

# Se instancia el helper
#ih = Helper(dsn="impala_prob")

# Ejecutar una consulta
#df_news = ih.obtener_dataframe("SELECT * FROM proceso_cap_analit_y_gob_de_inf.News_analisis")
df_news = pd.read_csv("temp_save_results.csv")
df_news['date'] =  pd.to_datetime(df_news['date'], format='%d/%m/%Y')
df_news.sort_values(by='date', ascending = True, inplace = True)
df_news = df_news.reset_index() 

Precios_indices = pd.read_excel('Consolidado RICs ETFs Sectores.xlsx', 
                                engine='openpyxl', sheet_name='Precios índices (Valores)')
Precios_acciones = pd.read_excel('Consolidado RICs ETFs Sectores.xlsx', 
                                 engine='openpyxl', sheet_name='Precios acciones (Valores)').drop([0]).dropna()

Precios_acciones['DATE'] =  pd.to_datetime(Precios_acciones['DATE'], format='%Y-%m-%d %H:M:S')

name_ticket = {"META PLATFORMS A-META-NEWS": "META",
               "AMAZON.COM-AMZN-NEWS": "AMZN",
               "EXXON MOBIL-XOM-NEWS": "XOM",
               "PROCTER & GAMBLE-PG-NEWS": "PG",
               "RAYTHEON TECHNOLOGIES-RTX-NEWS": "RTX",
               "APPLE-AAPL-NEWS": "AAPL",
               "LINDE-LIN-NEWS": "LIN",
               "NEXTERA ENERGY-NEE-NEWS": "NEE",
               "BERKSHIRE HATHAWAY 'B'-BRK.A-NEWS": "BRK.B",
               "AMERICAN TOWER-AMT-NEWS": "AMT",
               "JOHNSON CONTROLS INTL.-JCI-NEWS": "JNJ"}

In [2]:
import math
math.isnan(df_news[(df_news["date"] > "2020-09-20") & (df_news["date"] < "2020-10-20")]["chatgpt_predict"].mean())

True

In [2]:
import plotly.express as px

df = df_news[df_news["search"] == "META PLATFORMS A-META-NEWS"]
df = df.groupby([df['date'].dt.date])["chatgpt_predict"].mean()
fig = px.line(df, title='sentimiento vs tiempo')
fig.show()

In [6]:
Precios_acciones = Precios_acciones[(Precios_acciones["DATE"] > "2021-05-03") & (Precios_acciones["DATE"] < "2023-05-01")]

fig = px.line(Precios_acciones, x= "DATE", y= "META", title='precio accion vs tiempo')
fig.show()

In [7]:
Tendencia = []
for i in range(len(Precios_acciones) - 1):
    valor = Precios_acciones.iloc[i]["META"] - Precios_acciones.iloc[i - 1]["META"]
    if valor > 1:
        Tendencia.append([Precios_acciones.iloc[i]["DATE"], 1])
    elif valor < 1:
        Tendencia.append([Precios_acciones.iloc[i]["DATE"], -1])
    else: 
        Tendencia.append([Precios_acciones.iloc[i]["DATE"], 0])

Tendencia = pd.DataFrame(Tendencia, columns=["DATE", "Tendencia"])

In [3]:
import pandas as pd
import math
dates_in_range = [x.split(' ') for x in pd.date_range("2021-05-01", "2023-05-01", freq="W").strftime("%Y-%m-%d").tolist()]
dates_in_range_2 = [x.split(' ') for x in pd.date_range("2021-03-01", "2023-05-01", freq="W").strftime("%Y-%m-%d").tolist()]
df = df_news[df_news["search"] == "META PLATFORMS A-META-NEWS"]
out = []
promedio_anterior = 0 
for i in range(len(dates_in_range) - 1):
    Acciones_2 = Precios_acciones[(Precios_acciones["DATE"] >= dates_in_range_2[i][0]) & (Precios_acciones["DATE"] < dates_in_range_2[i+1][0])]["META"].mean()
    Acciones = Precios_acciones[(Precios_acciones["DATE"] >= dates_in_range[i][0]) & (Precios_acciones["DATE"] < dates_in_range[i+1][0])]["META"].mean()
    Sentimiento = df[(df["date"] >= dates_in_range[i][0]) & (df["date"] < dates_in_range[i+1][0])]["chatgpt_predict"].mean()
    valor = Acciones_2 - Acciones
    if valor > 1:
        Tendencia = 1
    elif valor < 1:
        Tendencia = -1
    else: 
        Tendencia = 0
    if math.isnan(Sentimiento):
        Sentimiento = promedio_anterior
    promedio_anterior = Sentimiento
    out.append([Acciones, Tendencia, Sentimiento, dates_in_range[i+1][0]])
out = pd.DataFrame(out, columns=["Acciones", "Tendencia precio", "Sentimiento", "date"])

In [7]:
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter(x=out["date"], y=out["Sentimiento"],
                    mode='lines',
                    name='META Sentimientos'))
fig.add_trace(go.Scatter(x=out["date"], y=out["Tendencia precio"],
                    mode='lines+markers',
                    name='Precio accion'))

fig.update_layout(
    title="Sentimeinto analisis",
    xaxis_title="Tiempo",
    yaxis_title="Sentimiento",
)

fig.show()

In [11]:
df[(df["Year"] == 2022) & (df["Month"] == 1)]["link"]

538    https://www.itnews.com.au/news/meta-platforms-...
572    https://www.bbc.com/news/world-us-canada-60091899
577    https://www.theverge.com/2022/1/24/22898651/me...
579    https://about.fb.com/news/2022/01/introducing-...
580    https://www.wsj.com/articles/meta-unveils-new-...
584    https://www.ft.com/content/e237df96-7cc1-44e5-...
585    https://www.investopedia.com/meta-diem-project...
589    https://www.bloomberg.com/news/articles/2022-0...
591    https://www.proactiveinvestors.co.uk/companies...
594    https://about.fb.com/news/2022/01/updates-to-a...
Name: link, dtype: object